# Frame OCR Extractor v1

Đọc keyframe từ GCS manifest, detect vùng chữ bằng PaddleOCR, nhận dạng chữ bằng VietOCR, rồi upload annotation OCR lên GCS.

Output được ghi lên GCS trước dưới dạng JSONL + run artifacts, chưa upload trực tiếp vào Supabase.


## 1. Parameters

**Note:** Đây là cell chính cần chỉnh trên Kaggle. Sau mỗi lần đổi bucket, batch, model, workers hoặc batch size, hãy chạy lại cell này trước các cell phía dưới.


In [ ]:
from types import SimpleNamespace

# GCS and dataset identity. Empty bucket reads Kaggle Secret GCS_BUCKET.
GCS_BUCKET = "aic_ai_2026"
GCS_BUCKET_SECRET_NAME = "GCS_BUCKET"
GCS_CREDENTIALS_FILE = ""
GCS_CREDENTIALS_JSON_SECRET_NAME = "GCS_CREDENTIALS_JSON"

DATASET_ID = "ai_challenge_2025"
PROFILE_VERSION = "autoshot_v1"
BATCHES = ["L21"]

# Frame extraction layout on GCS.
KEYFRAMES_PREFIX = "processed/keyframes"
MANIFESTS_PREFIX = "processed/keyframes_manifests"
INPUT_MANIFEST_URI = ""  # Optional explicit gs://.../shot_segments.csv or frames_manifest.jsonl.

# Extractor output layout on GCS.
OUTPUT_PREFIX = "features/extractors"
EXTRACTOR_VERSION = "fe-ocr-v1"
ANNOTATION_VERSION = "fe-ocr-v1"

# Kaggle local runtime.
RUN_ROOT = "/kaggle/working/feature_extractor_runs"
SCRATCH_DIR = "/kaggle/working/feature_extractor_scratch"
CLEANUP_LOCAL_FRAMES_AFTER_RUN = True

# Execution controls.
DRY_RUN_MAX_FRAMES = 20
DEMO_BATCHES = ["L21"]
DEMO_MAX_FRAMES = 64
FULL_MAX_FRAMES = None
CONFIRM_FULL_RUN = ""  # Set to RUN_FULL_DATASET before full run.

# Resume and failure behavior.
UPLOAD_TO_GCS = True
UPLOAD_RUN_ARTIFACTS = True
SKIP_EXISTING = True
OVERWRITE = False
RESUME_ANNOTATIONS_URI = ""  # Optional gs://.../annotations.jsonl used when SKIP_EXISTING=True.
FAIL_FAST = False

# Parallelism/progress. Tune these for Kaggle GPU/CPU size.
DOWNLOAD_WORKERS = 8
UPLOAD_WORKERS = 8
PIPELINE_BATCH_SIZE = 64
LOG_EVERY_N_FRAMES = 128
USE_TQDM = True

# Task-specific settings are below.
TEXT_DETECTION_MODEL = "PP-OCRv5_mobile_det"
VIETOCR_MODEL = "vgg_seq2seq"
OCR_LIMIT_SIDE_LEN = 960
OCR_LINE_Y_THRESHOLD = 35
OCR_LINE_X_GAP = 180
OCR_CROP_PAD = 12
OCR_DET_BATCH_SIZE = 16
OCR_RECOG_BATCH_SIZE = 64
OCR_DETECTOR_DEVICE = "cpu"  # Keep PaddleOCR on CPU to avoid Paddle GPU / PyTorch NCCL conflicts on Kaggle.
DEVICE = "auto"  # VietOCR still uses CUDA through PyTorch when available.

cfg = SimpleNamespace(**{name: value for name, value in globals().copy().items() if name.isupper() and not name.startswith("_")})
print("Parameters loaded for Frame OCR Extractor v1.")
print("Batches:", cfg.BATCHES, "Demo:", cfg.DEMO_BATCHES, "Upload:", cfg.UPLOAD_TO_GCS)

EXTRACTOR_NAME = "ocr"
cfg.EXTRACTOR_NAME = EXTRACTOR_NAME


## 2. Install Dependencies

**Note:** Chạy cell cài đặt một lần sau khi mở Kaggle session. Nếu Kaggle tải package/model từ Internet, bật Internet trong Notebook Settings.


In [ ]:
%pip install -q google-cloud-storage pandas tqdm numpy opencv-python-headless
%pip install -q --force-reinstall --no-cache-dir "pillow>=10.4,<12"
%pip install -q paddlepaddle==3.2.2 paddleocr vietocr
%pip install -q --force-reinstall --no-cache-dir "pillow>=10.4,<12"


## 3. Shared GCS, Manifest, Run Helpers

**Note:** Cell này chứa helper chung để đọc manifest từ GCS, tải frame, ghi artifact và upload kết quả. Nếu sửa helper, chạy lại cell này trước khi chạy dry/demo/full.


In [ ]:
from __future__ import annotations

import csv
import json
import logging
import os
import shutil
import time
import uuid
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass
from datetime import datetime, timezone
from io import StringIO
from pathlib import Path
from typing import Any, Iterable

import numpy as np
import pandas as pd
from tqdm.auto import tqdm


@dataclass
class RunLayout:
    """Local and GCS paths for one extractor run."""
    run_id: str
    run_dir: Path
    frames_dir: Path
    artifacts_dir: Path
    output_prefix: str
    annotations_path: Path
    errors_path: Path
    metrics_path: Path
    summary_path: Path
    log_path: Path


def utc_now() -> str:
    """Return an ISO-8601 UTC timestamp."""
    return datetime.now(timezone.utc).isoformat().replace("+00:00", "Z")


def cfg_value(config: Any, name: str, default: Any = None) -> Any:
    """Read a value from a SimpleNamespace-like config object."""
    return getattr(config, name, default)


def normalize_prefix(value: str) -> str:
    """Normalize a GCS object prefix without leading/trailing slashes."""
    return str(value or "").strip().strip("/")


def make_run_id(kind: str) -> str:
    """Create a unique, sortable run identifier."""
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    return f"{kind}_{stamp}_{uuid.uuid4().hex[:8]}"


def read_kaggle_secret(secret_name: str) -> str:
    """Read a Kaggle secret if the notebook is running on Kaggle."""
    if not secret_name:
        return ""
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(secret_name) or ""
    except Exception:
        return ""


def resolve_bucket_name(config: Any, require: bool = True) -> str:
    """Resolve the target GCS bucket from params, env vars, or Kaggle Secrets."""
    configured = str(cfg_value(config, "GCS_BUCKET", "") or "").strip()
    env_value = os.environ.get("GCS_BUCKET", "").strip()
    secret_name = str(cfg_value(config, "GCS_BUCKET_SECRET_NAME", "GCS_BUCKET") or "").strip()
    resolved = configured or env_value or read_kaggle_secret(secret_name).strip()
    if resolved.startswith("gs://"):
        resolved = resolved[len("gs://"):].split("/", 1)[0]
    if require and not resolved:
        raise RuntimeError("Set GCS_BUCKET in params, env vars, or Kaggle Secrets.")
    return resolved


def make_storage_client(config: Any):
    """Create a google-cloud-storage client using Kaggle Secrets or ADC."""
    from google.cloud import storage

    credentials_file = str(cfg_value(config, "GCS_CREDENTIALS_FILE", "") or os.environ.get("GCS_CREDENTIALS_FILE", "")).strip()
    credentials_json = os.environ.get("GCS_CREDENTIALS_JSON", "").strip()
    secret_name = str(cfg_value(config, "GCS_CREDENTIALS_JSON_SECRET_NAME", "GCS_CREDENTIALS_JSON") or "").strip()
    credentials_json = credentials_json or read_kaggle_secret(secret_name).strip()
    if credentials_json:
        from google.oauth2 import service_account
        credentials = service_account.Credentials.from_service_account_info(json.loads(credentials_json))
        return storage.Client(project=credentials.project_id, credentials=credentials)
    if credentials_file:
        return storage.Client.from_service_account_json(credentials_file)
    return storage.Client()


def parse_gcs_uri(uri: str) -> tuple[str, str]:
    """Parse gs://bucket/object into bucket and object name."""
    if not str(uri).startswith("gs://"):
        raise ValueError(f"Expected gs:// URI, got {uri}")
    bucket, _, blob = str(uri)[5:].partition("/")
    if not bucket or not blob:
        raise ValueError(f"Invalid GCS URI: {uri}")
    return bucket, blob


def make_output_prefix(config: Any, batch_id: str, run_id: str) -> str:
    """Build the task output prefix for one logical batch/run."""
    return (
        f"{normalize_prefix(cfg_value(config, 'OUTPUT_PREFIX', 'features/extractors'))}/"
        f"dataset={cfg_value(config, 'DATASET_ID')}/batch={batch_id}/"
        f"frame_profile={cfg_value(config, 'PROFILE_VERSION')}/"
        f"extractor={cfg_value(config, 'EXTRACTOR_NAME')}/"
        f"extractor_version={cfg_value(config, 'EXTRACTOR_VERSION')}/"
        f"run_id={run_id}/"
    )


def make_run_layout(config: Any, batch_id: str, run_kind: str) -> RunLayout:
    """Create local directories and output files for a run."""
    run_id = make_run_id(run_kind)
    run_dir = Path(str(cfg_value(config, "RUN_ROOT", "/kaggle/working/feature_extractor_runs"))) / run_id
    frames_dir = run_dir / "frames"
    artifacts_dir = run_dir / "artifacts"
    frames_dir.mkdir(parents=True, exist_ok=True)
    artifacts_dir.mkdir(parents=True, exist_ok=True)
    return RunLayout(run_id, run_dir, frames_dir, artifacts_dir, make_output_prefix(config, batch_id, run_id), artifacts_dir / "annotations.jsonl", artifacts_dir / "errors.jsonl", artifacts_dir / "metrics.csv", artifacts_dir / "summary.json", run_dir / "run.log")


def setup_logging(layout: RunLayout, verbose: bool = False) -> logging.Logger:
    """Configure console and file logging for a notebook run."""
    logger = logging.getLogger(str(cfg_value(cfg, "EXTRACTOR_NAME", "feature_extractor")))
    logger.setLevel(logging.DEBUG if verbose else logging.INFO)
    logger.handlers.clear()
    fmt = logging.Formatter("%(asctime)s %(levelname)s %(message)s")
    stream = logging.StreamHandler()
    stream.setFormatter(fmt)
    file_handler = logging.FileHandler(layout.log_path, encoding="utf-8")
    file_handler.setFormatter(fmt)
    logger.addHandler(stream)
    logger.addHandler(file_handler)
    return logger


def upload_file(bucket: Any, local_path: Path, object_key: str, content_type: str = "application/octet-stream") -> None:
    """Upload one local file to GCS."""
    bucket.blob(object_key).upload_from_filename(str(local_path), content_type=content_type, timeout=900)


def upload_text(bucket: Any, text: str, object_key: str, content_type: str = "text/plain") -> None:
    """Upload text content to GCS."""
    bucket.blob(object_key).upload_from_string(text, content_type=content_type, timeout=300)


def write_json(path: Path, payload: dict[str, Any]) -> None:
    """Write a JSON object to disk with UTF-8 encoding."""
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")


def append_jsonl(path: Path, records: Iterable[dict[str, Any]]) -> int:
    """Append JSONL records to a local file and return the row count."""
    count = 0
    with path.open("a", encoding="utf-8") as handle:
        for record in records:
            handle.write(json.dumps(record, ensure_ascii=False) + "\n")
            count += 1
    return count


def append_metric(path: Path, row: dict[str, Any]) -> None:
    """Append one row to metrics.csv, creating the header if needed."""
    exists = path.exists()
    with path.open("a", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(row.keys()))
        if not exists:
            writer.writeheader()
        writer.writerow(row)


def latest_blob_name(bucket: Any, prefix: str, suffix: str) -> str | None:
    """Return the latest blob under prefix with the requested suffix."""
    blobs = [blob for blob in bucket.list_blobs(prefix=prefix) if blob.name.endswith(suffix)]
    if not blobs:
        return None
    blobs.sort(key=lambda b: (b.updated or datetime.min.replace(tzinfo=timezone.utc), b.name), reverse=True)
    return blobs[0].name


def find_manifest_blobs(config: Any, bucket: Any, batches: list[str]) -> list[str]:
    """Find GCS manifest files for selected batches."""
    explicit = str(cfg_value(config, "INPUT_MANIFEST_URI", "") or "").strip()
    if explicit:
        parsed_bucket, blob_name = parse_gcs_uri(explicit)
        if parsed_bucket != bucket.name:
            raise ValueError(f"INPUT_MANIFEST_URI bucket {parsed_bucket} does not match {bucket.name}")
        return [blob_name]
    blobs: list[str] = []
    base = normalize_prefix(cfg_value(config, "MANIFESTS_PREFIX", "processed/keyframes_manifests"))
    for batch_id in batches:
        prefix = f"{base}/dataset={cfg_value(config, 'DATASET_ID')}/batch={batch_id}/profile={cfg_value(config, 'PROFILE_VERSION')}/"
        name = latest_blob_name(bucket, prefix, "shot_segments.csv") or latest_blob_name(bucket, prefix, "frames_manifest.jsonl")
        if name is None:
            raise FileNotFoundError(f"No shot_segments.csv or frames_manifest.jsonl found under gs://{bucket.name}/{prefix}")
        blobs.append(name)
    return blobs


def read_manifest_blob(bucket: Any, blob_name: str) -> pd.DataFrame:
    """Read a CSV or JSONL frame manifest from GCS into a DataFrame."""
    text = bucket.blob(blob_name).download_as_text(timeout=900)
    if blob_name.endswith(".csv"):
        return pd.read_csv(StringIO(text))
    rows = [json.loads(line) for line in text.splitlines() if line.strip()]
    return pd.DataFrame(rows)


def normalize_manifest_records(df: pd.DataFrame, config: Any, batch_id: str | None = None) -> list[dict[str, Any]]:
    """Normalize frame manifest columns into the extractor record contract."""
    if df.empty:
        return []
    df = df.copy()
    if "saved" in df.columns:
        df = df[df["saved"].astype(str).str.lower().isin(["true", "1", "yes"])]
    records: list[dict[str, Any]] = []
    for _, row in df.iterrows():
        image_gcs_uri = str(row.get("image_gcs_uri") or row.get("gcs_uri") or row.get("image_uri") or "").strip()
        if not image_gcs_uri:
            continue
        video_id = str(row.get("video_id") or Path(image_gcs_uri).parent.name).strip()
        frame_idx = int(float(row.get("frame_idx", 0) or 0))
        keyframe_id = str(row.get("keyframe_id") or f"{video_id}_F{frame_idx:06d}")
        records.append({
            "dataset_id": str(row.get("dataset_id") or cfg_value(config, "DATASET_ID")),
            "batch_id": str(row.get("batch_id") or batch_id or "").strip(),
            "video_id": video_id,
            "video_name": str(row.get("video_name") or ""),
            "shot_id": str(row.get("shot_id") or ""),
            "shot_start_frame": int(float(row.get("shot_start_frame", 0) or 0)),
            "shot_end_frame": int(float(row.get("shot_end_frame", 0) or 0)),
            "frame_type": str(row.get("frame_type") or ""),
            "frame_idx": frame_idx,
            "frame_sec": float(row.get("frame_sec", row.get("timestamp", 0)) or 0),
            "timestamp_ms": int(float(row.get("frame_sec", 0) or 0) * 1000),
            "keyframe_id": keyframe_id,
            "image_rel_path": str(row.get("image_rel_path") or f"{video_id}/{Path(image_gcs_uri).name}"),
            "image_gcs_uri": image_gcs_uri,
            "image_storage_key": str(row.get("image_storage_key") or parse_gcs_uri(image_gcs_uri)[1]),
            "fps": float(row.get("fps", 0) or 0),
            "profile_version": str(row.get("profile_version") or cfg_value(config, "PROFILE_VERSION")),
        })
    return records


def discover_frame_records(config: Any, bucket: Any, batches: list[str], max_frames: int | None = None) -> list[dict[str, Any]]:
    """Discover and normalize frame records for selected logical batches."""
    all_records: list[dict[str, Any]] = []
    for blob_name in find_manifest_blobs(config, bucket, batches):
        batch_hint = next((part.split("=", 1)[1] for part in blob_name.split("/") if part.startswith("batch=")), None)
        all_records.extend(normalize_manifest_records(read_manifest_blob(bucket, blob_name), config, batch_hint))
    all_records.sort(key=lambda r: (r["batch_id"], r["video_id"], r["frame_idx"], r["keyframe_id"]))
    return all_records[: int(max_frames)] if max_frames is not None else all_records


def local_frame_path(layout: RunLayout, record: dict[str, Any]) -> Path:
    """Return the local scratch path for one frame record."""
    return layout.frames_dir / record["video_id"] / Path(record["image_gcs_uri"]).name


def download_one_frame(record: dict[str, Any], layout: RunLayout, client: Any) -> dict[str, Any]:
    """Download one frame from GCS into local scratch and return an updated record."""
    started = time.perf_counter()
    bucket_name, blob_name = parse_gcs_uri(record["image_gcs_uri"])
    out_path = local_frame_path(layout, record)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    if not out_path.exists() or out_path.stat().st_size == 0:
        client.bucket(bucket_name).blob(blob_name).download_to_filename(str(out_path), timeout=900)
    updated = dict(record)
    updated["local_image_path"] = str(out_path)
    updated["download_ms"] = int((time.perf_counter() - started) * 1000)
    return updated


def download_frames(records: list[dict[str, Any]], layout: RunLayout, client: Any, config: Any) -> list[dict[str, Any]]:
    """Download many frames concurrently from GCS."""
    downloaded: list[dict[str, Any]] = []
    workers = max(1, int(cfg_value(config, "DOWNLOAD_WORKERS", 8)))
    with ThreadPoolExecutor(max_workers=workers, thread_name_prefix="gcs-download") as pool:
        futures = [pool.submit(download_one_frame, record, layout, client) for record in records]
        for future in tqdm(as_completed(futures), total=len(futures), desc="Downloading frames", disable=not cfg_value(config, "USE_TQDM", True)):
            downloaded.append(future.result())
    downloaded.sort(key=lambda r: (r["batch_id"], r["video_id"], r["frame_idx"], r["keyframe_id"]))
    return downloaded


def iter_batches(items: list[Any], batch_size: int) -> Iterable[list[Any]]:
    """Yield fixed-size batches from a list."""
    batch_size = max(1, int(batch_size))
    for start in range(0, len(items), batch_size):
        yield items[start:start + batch_size]


def base_annotation(record: dict[str, Any], config: Any, kind: str, run_id: str) -> dict[str, Any]:
    """Create the shared frame annotation JSON payload."""
    return {
        "dataset_id": record["dataset_id"], "batch_id": record["batch_id"], "video_id": record["video_id"],
        "keyframe_id": record["keyframe_id"], "frame_id": record["keyframe_id"], "shot_id": record.get("shot_id", ""),
        "frame_idx": record["frame_idx"], "frame_sec": record["frame_sec"], "timestamp_ms": record.get("timestamp_ms", int(float(record.get("frame_sec", 0)) * 1000)),
        "frame_type": record.get("frame_type", ""), "image_gcs_uri": record["image_gcs_uri"], "image_storage_key": record.get("image_storage_key", ""),
        "kind": kind, "caption": None, "ocr_texts": [], "detected_objects": [], "object_counts": {}, "detections": [],
        "text_value": None, "json_value": {}, "confidence": 1.0, "model_version": str(cfg_value(config, "MODEL_VERSION", "unknown")),
        "annotation_version": str(cfg_value(config, "ANNOTATION_VERSION", cfg_value(config, "EXTRACTOR_VERSION", "v1"))), "run_id": run_id, "created_at": utc_now(),
    }


def upload_standard_artifacts(bucket: Any, layout: RunLayout, success: bool) -> None:
    """Upload standard run artifacts and optional _SUCCESS marker to GCS."""
    for path, content_type in [(layout.annotations_path, "application/jsonl"), (layout.errors_path, "application/jsonl"), (layout.metrics_path, "text/csv"), (layout.summary_path, "application/json"), (layout.log_path, "text/plain")]:
        if path.exists():
            upload_file(bucket, path, layout.output_prefix + path.name, content_type)
    if success:
        upload_text(bucket, "", layout.output_prefix + "_SUCCESS", "text/plain")



def output_base_prefix(config: Any, batch_id: str) -> str:
    """Build the extractor output prefix without run_id for resume discovery."""
    return (
        f"{normalize_prefix(cfg_value(config, 'OUTPUT_PREFIX', 'features/extractors'))}/"
        f"dataset={cfg_value(config, 'DATASET_ID')}/batch={batch_id}/"
        f"frame_profile={cfg_value(config, 'PROFILE_VERSION')}/"
        f"extractor={cfg_value(config, 'EXTRACTOR_NAME')}/"
        f"extractor_version={cfg_value(config, 'EXTRACTOR_VERSION')}/"
    )


def find_resume_annotations_blob(config: Any, bucket: Any, batches: list[str]) -> str | None:
    """Find an annotations.jsonl blob to use for resume filtering."""
    explicit = str(cfg_value(config, "RESUME_ANNOTATIONS_URI", "") or "").strip()
    if explicit:
        parsed_bucket, blob_name = parse_gcs_uri(explicit)
        if parsed_bucket != bucket.name:
            raise ValueError(f"RESUME_ANNOTATIONS_URI bucket {parsed_bucket} does not match {bucket.name}")
        return blob_name
    candidates = []
    for batch_id in batches:
        prefix = output_base_prefix(config, batch_id)
        candidates.extend([blob for blob in bucket.list_blobs(prefix=prefix) if blob.name.endswith("annotations.jsonl")])
    if not candidates:
        return None
    candidates.sort(key=lambda b: (b.updated or datetime.min.replace(tzinfo=timezone.utc), b.name), reverse=True)
    return candidates[0].name


def load_processed_keyframes(config: Any, bucket: Any, batches: list[str]) -> set[str]:
    """Load keyframe IDs already present in a previous successful annotations JSONL."""
    if not cfg_value(config, "SKIP_EXISTING", True) or cfg_value(config, "OVERWRITE", False):
        return set()
    blob_name = find_resume_annotations_blob(config, bucket, batches)
    if not blob_name:
        return set()
    text = bucket.blob(blob_name).download_as_text(timeout=900)
    processed: set[str] = set()
    for line in text.splitlines():
        if not line.strip():
            continue
        try:
            row = json.loads(line)
        except Exception:
            continue
        if not row.get("error") and row.get("keyframe_id"):
            processed.add(str(row["keyframe_id"]))
    return processed

def dry_run(config: Any, max_frames: int | None = None) -> dict[str, Any]:
    """Discover frame records without downloading images or running models."""
    client = make_storage_client(config)
    bucket = client.bucket(resolve_bucket_name(config, require=True))
    batches = [str(batch).upper() for batch in cfg_value(config, "BATCHES", [])]
    records = discover_frame_records(config, bucket, batches, max_frames=max_frames)
    return {"status": "DRY_RUN_OK", "bucket": bucket.name, "batches": batches, "planned_frames": len(records), "sample_records": records[:5]}


## 3b. Task Model And Extraction Logic

**Note:** Cell này chứa model và logic riêng của extractor. Các hàm đều trả record theo contract JSONL trung gian, chưa ghi trực tiếp vào Supabase.


In [ ]:
def load_task_model(config: Any, logger: logging.Logger) -> dict[str, Any]:
    """Load Paddle text detector and VietOCR recognizer."""
    import torch
    from paddleocr import TextDetection
    from vietocr.tool.config import Cfg
    from vietocr.tool.predictor import Predictor
    device = "cuda" if torch.cuda.is_available() and cfg_value(config, "DEVICE", "auto") != "cpu" else "cpu"
    viet_cfg = Cfg.load_config_from_name(str(cfg_value(config, "VIETOCR_MODEL", "vgg_seq2seq")))
    viet_cfg["cnn"]["pretrained"] = False
    viet_cfg["device"] = device
    predictor = Predictor(viet_cfg)
    detector_device = str(cfg_value(config, "OCR_DETECTOR_DEVICE", "cpu"))
    detector = TextDetection(model_name=str(cfg_value(config, "TEXT_DETECTION_MODEL", "PP-OCRv5_mobile_det")), device=detector_device, limit_side_len=int(cfg_value(config, "OCR_LIMIT_SIDE_LEN", 960)), limit_type="max")
    config.MODEL_VERSION = f"{cfg_value(config, 'TEXT_DETECTION_MODEL')}-{cfg_value(config, 'VIETOCR_MODEL')}"
    logger.info("Loaded OCR detector=%s detector_device=%s recognizer=%s recognizer_device=%s", cfg_value(config, "TEXT_DETECTION_MODEL"), detector_device, cfg_value(config, "VIETOCR_MODEL"), device)
    return {"detector": detector, "predictor": predictor, "device": device, "detector_device": detector_device}


def _poly_to_xyxy(poly: Any) -> list[float]:
    """Convert a 4-point polygon to [x1, y1, x2, y2]."""
    poly = np.asarray(poly)
    return [float(np.min(poly[:, 0])), float(np.min(poly[:, 1])), float(np.max(poly[:, 0])), float(np.max(poly[:, 1]))]


def _merge_boxes_by_line(polys: list[Any], y_thresh: int = 35, x_gap_thresh: int = 180) -> list[list[float]]:
    """Merge nearby text detection boxes into approximate text lines."""
    boxes = sorted([_poly_to_xyxy(poly) for poly in polys], key=lambda box: (box[1], box[0]))
    lines: list[dict[str, list[float]]] = []
    for box in boxes:
        x1, y1, x2, y2 = box
        cy = (y1 + y2) / 2
        matched = False
        for line in lines:
            lx1, ly1, lx2, ly2 = line["box"]
            lcy = (ly1 + ly2) / 2
            if abs(cy - lcy) < y_thresh and x1 - lx2 < x_gap_thresh:
                line["box"] = [min(lx1, x1), min(ly1, y1), max(lx2, x2), max(ly2, y2)]
                matched = True
                break
        if not matched:
            lines.append({"box": box})
    return [line["box"] for line in lines]


def _crop_xyxy(img_rgb: np.ndarray, box: list[float], pad: int = 12) -> np.ndarray:
    """Crop a padded [x1, y1, x2, y2] region from an RGB image."""
    h, w = img_rgb.shape[:2]
    x1, y1, x2, y2 = map(int, box)
    x1, y1 = max(0, x1 - pad), max(0, y1 - pad)
    x2, y2 = min(w, x2 + pad), min(h, y2 + pad)
    return img_rgb[y1:y2, x1:x2]


def _extract_polys(det_result: Any) -> list[np.ndarray]:
    """Extract PaddleOCR polygons from a detection result object."""
    boxes = det_result.get("dt_polys", []) if isinstance(det_result, dict) else getattr(det_result, "dt_polys", [])
    return [np.asarray(box, dtype=np.float32) for box in boxes if np.asarray(box).shape == (4, 2)]


def extract_task_batch(records: list[dict[str, Any]], ctx: dict[str, Any], config: Any, layout: RunLayout) -> list[dict[str, Any]]:
    """Extract OCR annotations for a frame batch."""
    import cv2
    from PIL import Image
    paths = [record["local_image_path"] for record in records]
    try:
        det_results = list(ctx["detector"].predict(paths))
    except Exception:
        det_results = []
        for path in paths:
            det_results.extend(list(ctx["detector"].predict(path)))
    outputs = []
    for record, det_result in zip(records, det_results):
        img = cv2.imread(record["local_image_path"])
        texts, detections = [], []
        if img is not None:
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            boxes = _merge_boxes_by_line(_extract_polys(det_result), int(cfg_value(config, "OCR_LINE_Y_THRESHOLD", 35)), int(cfg_value(config, "OCR_LINE_X_GAP", 180)))
            crops, crop_boxes = [], []
            for box in boxes:
                crop = _crop_xyxy(img_rgb, box, pad=int(cfg_value(config, "OCR_CROP_PAD", 12)))
                if crop is not None and crop.shape[0] >= 5 and crop.shape[1] >= 5:
                    crops.append(Image.fromarray(crop))
                    crop_boxes.append(box)
            raw_texts = [text.strip() for text in ctx["predictor"].predict_batch(crops)] if crops and hasattr(ctx["predictor"], "predict_batch") else ([ctx["predictor"].predict(crop).strip() for crop in crops] if crops else [])
            for text, box in zip(raw_texts, crop_boxes):
                if text:
                    texts.append(text)
                    detections.append({"text": text, "bbox_xyxy": [round(float(v), 2) for v in box], "confidence": 1.0})
        item = base_annotation(record, config, "ocr", layout.run_id)
        item.update({"ocr_texts": texts, "text_value": " ".join(texts) if texts else None, "detections": detections, "json_value": {"ocr_texts": texts, "detections": detections}})
        outputs.append(item)
    return outputs


def run_frame_extractor(config: Any, batches: list[str], max_frames: int | None, run_kind: str) -> dict[str, Any]:
    """Run a frame-level extractor and upload JSONL artifacts to GCS."""
    client = make_storage_client(config)
    bucket = client.bucket(resolve_bucket_name(config, require=True))
    layout = make_run_layout(config, batches[0] if len(batches) == 1 else "all", run_kind)
    logger = setup_logging(layout)
    started = time.perf_counter()
    records = discover_frame_records(config, bucket, batches, max_frames=max_frames)
    discovered_count = len(records)
    logger.info("Discovered %d frames for batches=%s", discovered_count, batches)
    processed_keys = load_processed_keyframes(config, bucket, batches)
    if processed_keys:
        before_count = len(records)
        records = [record for record in records if record["keyframe_id"] not in processed_keys]
        skipped = before_count - len(records)
        logger.info("Resume filter skipped %d already processed frames from previous annotations", skipped)
    else:
        skipped = 0
    remaining_count = len(records)
    records = download_frames(records, layout, client, config) if records else []
    model_ctx = load_task_model(config, logger)
    processed = 0
    failed = 0
    batch_size = int(cfg_value(config, "PIPELINE_BATCH_SIZE", 64))
    for batch_index, record_batch in enumerate(tqdm(list(iter_batches(records, batch_size)), desc=f"{cfg_value(config, 'EXTRACTOR_NAME')} batches", disable=not cfg_value(config, "USE_TQDM", True)), start=1):
        batch_started = time.perf_counter()
        try:
            output_records = extract_task_batch(record_batch, model_ctx, config, layout)
            append_jsonl(layout.annotations_path, output_records)
            processed += len(output_records)
            elapsed = time.perf_counter() - batch_started
            append_metric(layout.metrics_path, {"run_id": layout.run_id, "batch_index": batch_index, "frames": len(record_batch), "processed": len(output_records), "failed": 0, "seconds": round(elapsed, 3), "frames_per_second": round(len(record_batch) / elapsed, 4) if elapsed else 0})
        except Exception as exc:
            failed += len(record_batch)
            append_jsonl(layout.errors_path, [{**base_annotation(record, config, str(cfg_value(config, "EXTRACTOR_NAME")), layout.run_id), "error": str(exc)} for record in record_batch])
            append_metric(layout.metrics_path, {"run_id": layout.run_id, "batch_index": batch_index, "frames": len(record_batch), "processed": 0, "failed": len(record_batch), "seconds": round(time.perf_counter() - batch_started, 3), "frames_per_second": 0})
            logger.exception("Batch %d failed: %s", batch_index, exc)
            if cfg_value(config, "FAIL_FAST", False):
                raise
        total_seen = processed + failed + skipped
        if total_seen and total_seen % int(cfg_value(config, "LOG_EVERY_N_FRAMES", 128)) < batch_size:
            logger.info(
                "Progress: %d/%d frames, processed=%d skipped=%d failed=%d remaining_to_run=%d",
                total_seen,
                discovered_count,
                processed,
                skipped,
                failed,
                remaining_count,
            )
    success = failed == 0
    summary = {"run_id": layout.run_id, "status": "SUCCESS" if success else "COMPLETED_WITH_ERRORS", "extractor": cfg_value(config, "EXTRACTOR_NAME"), "extractor_version": cfg_value(config, "EXTRACTOR_VERSION"), "dataset_id": cfg_value(config, "DATASET_ID"), "batches": batches, "planned_frames": discovered_count, "processed_frames": processed, "skipped_frames": skipped, "failed_frames": failed, "duration_seconds": round(time.perf_counter() - started, 3), "output_prefix": f"gs://{bucket.name}/{layout.output_prefix}", "created_at": utc_now()}
    write_json(layout.summary_path, summary)
    if cfg_value(config, "UPLOAD_TO_GCS", True) and cfg_value(config, "UPLOAD_RUN_ARTIFACTS", True):
        upload_standard_artifacts(bucket, layout, success)
    if cfg_value(config, "CLEANUP_LOCAL_FRAMES_AFTER_RUN", True):
        shutil.rmtree(layout.frames_dir, ignore_errors=True)
    return summary


def run_demo(config: Any) -> dict[str, Any]:
    """Run the extractor on a small selected frame sample."""
    return run_frame_extractor(config, [str(batch).upper() for batch in cfg_value(config, "DEMO_BATCHES", ["L21"])], cfg_value(config, "DEMO_MAX_FRAMES", 64), "demo")


def run_full(config: Any) -> list[dict[str, Any]]:
    """Run the extractor on all selected batches with a safety guard."""
    if cfg_value(config, "CONFIRM_FULL_RUN", "") != "RUN_FULL_DATASET":
        print('Skipped full run. Set CONFIRM_FULL_RUN = "RUN_FULL_DATASET" in the parameter cell and rerun it.')
        return []
    return [run_frame_extractor(config, [str(batch).upper()], cfg_value(config, "FULL_MAX_FRAMES", None), f"full_{str(batch).lower()}") for batch in cfg_value(config, "BATCHES", [])]


## 4. Dry Run

**Note:** Cell này chỉ kiểm tra credential, manifest discovery và planned GCS paths; không download frame, không load model, không upload output.


In [ ]:
dry_summary = dry_run(cfg, max_frames=cfg.DRY_RUN_MAX_FRAMES)
dry_summary


## 5. Demo Run

**Note:** Cell này chạy end-to-end trên một mẫu nhỏ (`DEMO_BATCHES`, `DEMO_MAX_FRAMES`) và upload artifact nếu `UPLOAD_TO_GCS=True`.


In [ ]:
demo_summary = run_demo(cfg)
demo_summary


## 6. Full Run

**Note:** Cell này được khóa an toàn. Chỉ chạy toàn bộ khi bạn đặt `CONFIRM_FULL_RUN = "RUN_FULL_DATASET"` trong parameter cell rồi chạy lại parameter cell.


In [ ]:
full_summaries = run_full(cfg)
full_summaries


## 7. Inspect Latest Local Artifacts

**Note:** Cell này liệt kê các artifact mới nhất trong `RUN_ROOT` để kiểm tra nhanh trước khi kết thúc Kaggle session.


In [ ]:
run_root = Path(cfg.RUN_ROOT)
latest = sorted([path for path in run_root.glob("*") if path.is_dir()], key=lambda path: path.stat().st_mtime, reverse=True)[:5]
if not latest:
    print(f"No runs found under {run_root}")
for path in latest:
    print(path)
    for artifact in ["artifacts/summary.json", "artifacts/annotations.jsonl", "artifacts/errors.jsonl", "artifacts/metrics.csv", "run.log"]:
        candidate = path / artifact
        if candidate.exists():
            print("  ", candidate, candidate.stat().st_size, "bytes")
